# Storybook Studio — Google Colab GPU Backend

Run free AI inference for text, image, and video generation using Colab's free GPU.

After running all cells, set the output URL as `COLAB_ENDPOINT` in your `.env` file.

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers diffusers accelerate gradio fastapi uvicorn
!pip install -q sentencepiece bitsandbytes xformers
!pip install -q pyngrok

In [ ]:
import torch
import gradio as gr
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import threading
import json
import base64
from io import BytesIO

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading Mistral 7B for text generation...")
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True
)
print("Text model loaded!")

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch

print("Loading Stable Diffusion XL for image generation...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    safety_checker=None,
    requires_safety_checker=False,
)
pipe.to("cuda")
print("Image model loaded!")

In [ ]:
# FastAPI server
app = FastAPI()

class TextRequest(BaseModel):
    prompt: str
    system_prompt: str = ""
    max_tokens: int = 2048
    temperature: float = 0.85

class ImageRequest(BaseModel):
    prompt: str
    negative_prompt: str = ""
    num_steps: int = 30
    guidance_scale: float = 7.5

class VideoRequest(BaseModel):
    prompt: str
    image_url: str = ""
    duration: int = 5
    motion_strength: float = 0.7

@app.get("/health")
async def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}

@app.post("/generate/text")
async def generate_text(req: TextRequest):
    full_prompt = f"<s>[INST] {req.system_prompt}\n\n{req.prompt} [/INST]" if req.system_prompt else req.prompt
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=req.max_tokens,
        temperature=req.temperature,
        do_sample=True,
        top_p=0.95,
        repetition_penalty=1.1
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip input prompt from output
    if req.prompt in text:
        text = text.split(req.prompt)[-1].strip()
    return {"text": text, "model": "Mistral-7B"}

@app.post("/generate/image")
async def generate_image(req: ImageRequest):
    result = pipe(
        prompt=req.prompt,
        negative_prompt=req.negative_prompt,
        num_inference_steps=req.num_steps,
        guidance_scale=req.guidance_scale,
    ).images[0]
    buffered = BytesIO()
    result.save(buffered, format="PNG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode()
    return {"image": img_b64}

@app.post("/generate/video")
async def generate_video(req: VideoRequest):
    # Placeholder — CogVideoX requires significant VRAM
    # For production, use HuggingFace inference API or a dedicated notebook
    return {"video_url": "", "note": "CogVideoX requires 24GB+ VRAM. Use HuggingFace inference API instead."}

In [ ]:
# Start server and expose via Gradio share link
import threading
import time

# Run FastAPI in a background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=7860, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("Server started on port 7860")

In [ ]:
# Create a Gradio share link for external access
import gradio as gr

# Quick health check demo
def check_health():
    import requests
    r = requests.get("http://localhost:7860/health")
    return r.json()

def generate_text_demo(prompt, system, max_tokens, temp):
    import requests
    r = requests.post("http://localhost:7860/generate/text", json={
        "prompt": prompt, "system_prompt": system,
        "max_tokens": max_tokens, "temperature": temp
    })
    return r.json().get("text", "")

def generate_image_demo(prompt, neg_prompt):
    import requests
    import base64
    from PIL import Image
    from io import BytesIO
    r = requests.post("http://localhost:7860/generate/image", json={
        "prompt": prompt, "negative_prompt": neg_prompt
    })
    data = r.json().get("image", "")
    if data:
        return Image.open(BytesIO(base64.b64decode(data)))
    return None

with gr.Blocks(title="Storybook Colab Backend") as demo:
    gr.Markdown("# Storybook Studio — Colab GPU Backend")
    gr.Markdown(f"**GPU:** {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
    
    with gr.Tab("Text Generation"):
        with gr.Row():
            prompt = gr.Textbox(label="Prompt", lines=5)
            system = gr.Textbox(label="System Prompt", lines=3)
        max_tokens = gr.Slider(100, 4096, 1024, label="Max Tokens")
        temp = gr.Slider(0.1, 2.0, 0.85, label="Temperature")
        btn = gr.Button("Generate")
        output = gr.Textbox(label="Output", lines=10)
        btn.click(generate_text_demo, [prompt, system, max_tokens, temp], output)
    
    with gr.Tab("Image Generation"):
        with gr.Row():
            img_prompt = gr.Textbox(label="Prompt", lines=3)
            img_neg = gr.Textbox(label="Negative Prompt", lines=2)
        img_btn = gr.Button("Generate Image")
        img_output = gr.Image(label="Result")
        img_btn.click(generate_image_demo, [img_prompt, img_neg], img_output)

# Launch with public share link
print("=" * 60)
print("LAUNCHING GRADIO SHARE LINK...")
print("=" * 60)
print("")
print("Copy the URL below and set it as COLAB_ENDPOINT in your .env file:")
print("")
demo.launch(share=True, server_port=7861)
print("")
print("NOTE: Keep this Colab tab open while using Storybook Studio!")